# WuWa Mod Hash Fixer
[![Static Badge](https://img.shields.io/badge/Jupyter_Notebook-F37726?style=for-the-badge)](https://jupyter.org/)

<br>

**Your WuWa mod's textures look broken? Its hashes have probably gone stale, and nothing is corrupt.**

WWMI binds a mod's textures by hash -- `[TextureOverrideTexture<N>] hash = <h>`. Wuthering Waves
**rehashes** a texture between game versions, so once the hash the author exported no longer matches
anything the game emits, the override never fires and the surface draws with the **game's own art**
instead of the mod's. The mod is addressing textures that no longer exist under those names.

This notebook resolves each stale hash **backwards** to the role it played (`upperDiffuse`,
`hairNormal`, ...) and then **forwards** to the hash that role has on the current version, using the
hash history AGRemap carries for every character it knows. It is the Wuthering Waves counterpart of
what ORFix does for Genshin every version, and it needs no frame dump.

It **reports first and writes nothing**, keeps one backup per `.ini`, and can undo itself.

<br>


First install what the tool needs


In [ ]:
%pip install -r ../requirements.txt


<br>

Then choose how to install AGRemap's API

**Option A**: install it from [Pypi](https://pypi.org/project/AnimeGameRemap/)


In [ ]:
%pip install -U AnimeGameRemap


In [ ]:
import AnimeGameRemap as AGR


<br>

**Option B**: Alternatively, import the API locally from a checkout of the repo


In [ ]:
import os
import sys

# Note: Make sure the path correctly points where the AGRemap's API is located
#   (it must end up absolute: the API's native extensions cannot be loaded from a relative sys.path entry)
sys.path.insert(1, os.path.abspath(r"../../../Anime Game Remap (for all users)/api/src/py"))


<br>

## Load the tool


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(r".."))
from ModHashFixer.ModHashFixer import ModHashFixer

print("characters this notebook knows:", ", ".join(sorted(ModHashFixer.characters())))


<br>

## Point it at your mod

`Mod` is the mod's folder -- every `.ini` under it is read, and anything named `DISABLED*` is skipped.

Leave `Character` as `None` to have it worked out from the hashes themselves (recommended: a mod
folder is named after whatever its author or the downloader called it, which tells you nothing).
Leave `Version` as `None` for the newest version the library knows.


In [ ]:
Mod = r"C:/Users/You/XXMI-Launcher/WWMI/Mods/SomeMod"    # <- the mod to fix
Character = None                                            # e.g. "Chisa"
Version = None                                              # e.g. "3.6"

fixer = ModHashFixer(Mod, character = Character, version = Version)
print(f"{len(fixer.files)} .ini file(s) found")
for f in fixer.files:
    print("   ", os.path.relpath(f, Mod))


<br>

## Optional: your own hashes, for a character this does not cover

The history above only covers the characters AGRemap has registered. **A character it does not
carry is not out of reach** --- if you already know which old hash became which new one, give it
the table and it will use it. Yours **wins** over the library, and anything resolved from it is
reported as `custom (yours)`, so you can always see which answers came from where.

Two ways to get such a table:

* **A community fixer's table.** The WuWa fixers publish `hash_maps.json` and similar. Pass the
  file(s) to `ModHashFixer.loadHashMaps(...)`; it takes every `"oldhash": "newhash"` pair it finds
  at any depth, because these are published with no agreed layout.
* **Your own pair.** Dump the character on the current version (3DMigoto's frame analysis), find the
  texture you care about, and pair its hash with the one the mod is using. One line per texture.

Write it out and re-run the cell above --- the plan will pick them up.


In [ ]:
# Option 1 -- load community tables (any nesting is handled)
# Extra = ModHashFixer.loadHashMaps(r"hash_maps.json", r"another_table.json")

# Option 2 -- write the pairs yourself: {the hash the MOD uses: the hash the GAME uses now}
Extra = {
    # "37250244": "f2646d21",
}

fixer = ModHashFixer(Mod, character = Character, version = Version, extraHashes = Extra)
print(f"{len(fixer.extraHashes)} of your own pair(s) loaded")


<br>

**Adding the character properly instead.** If you want the character covered for everyone rather
than in your own notebook, the lasting place for it is the library's `HashData` --- a row per
`(version, character, role)`. That is what turns a one-off table into something every later
version can be resolved through, and it is how the characters here got their history.
`Tools/Misc/Diagnostics/chisaHashHistory.py` shows the shape: it derives older generations from
mods themselves, and keeps a hash only on hash-level evidence rather than guessing from a picture.


<br>

## See what it would change

Nothing is written by this cell. Read the list before applying it:

* **to update** -- a hash that resolved to one of this character's roles and has moved since.
* **already current** -- resolved, and already right. A mod where everything is current does not have
  this problem, and whatever is wrong with it is something else.
* **geometry left alone** -- `vb0` / `cb4` / the shape-key pair. A mod whose `vb0` is stale does not
  draw **at all**, which is a different symptom; rewriting those on a mod that *does* draw breaks what
  works. Pass `geometry = True` to `ModHashFixer(...)` only if the mod is completely invisible.
* **unrecognised** -- not this character's, or older than the recorded history. Left alone, never
  guessed. If a texture you care about is in this list, its older hash is simply not in the library
  yet, and the tool cannot place it.


In [ ]:
name, scores = (Character, None) if Character else fixer.detect()
if name is None:
    raise SystemExit("no hash in this mod resolves to any character the library knows")

print(f"character: {name}" + ("" if not scores else f"   (detected from {scores})"))

changes, counts, unrecognised, newBytes = fixer.plan(ModHashFixer.characters()[name])
for rel, section, role, old, new in changes:
    print(f"  {rel} [{section}]  {role:22s} {old} -> {new}")

print(f"\n{len(changes)} to update, {counts['current']} already current, "
      f"{counts['geometry']} geometry left alone, {sum(unrecognised.values())} unrecognised")
for value, n in unrecognised.most_common(20):
    print(f"    unrecognised: {value} x{n}")


<br>

## Apply it

This writes the `.ini` files, keeping the original beside each one as `<name>.ini.hashfixBKUP`.
Line endings and encoding are preserved byte for byte.


In [ ]:
for path in fixer.write(newBytes):
    print("wrote", path)
print(f"{len(newBytes)} file(s) written")


<br>

## Undo it

Restores every `.ini` this tool backed up, and removes the backups.


In [ ]:
for path in fixer.undo():
    print("restored", path)


<br>

## If it did not help

A few things this tool deliberately does **not** do, so you know where to look next:

* it does not touch geometry hashes by default (see above);
* it cannot place a hash the library has never recorded -- the **unrecognised** list is the honest
  limit of its coverage, and extending it means adding rows to the library's `HashData`;
* it does not repair a mod that names a texture **file** which is not on disk. That is a different
  fault, and it shows up as a dangling `filename =` rather than a stale `hash =`;
* it changes nothing about geometry, weights, or shaders. If the mod's shape is wrong, or a part is
  missing, the hashes are not your problem.
